<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Tienda Ordenada o Plaza de Mercado: Almacén frente a Lago 🏪
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 02
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/02%20-%20Data%20Warehousing%20y%20Data%20Lake/Para%20Dummies/02_DW_vs_Lake_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 02 del Módulo 02 (Data Warehouse frente a Data Lake)](../02_Comparacion_Data_Warehouse_vs_Data_Lake.ipynb). Compara dos maneras de guardar información con dos lugares que conoces:

| La tienda ordenada (**almacén**) | La plaza de mercado (**lago**) |
|---|---|
| Cada producto tiene **su estante, su etiqueta y su formato**; **no entra nada que no encaje** | **Todo entra** tal como llega: costales, cajas, canastos |
| Encuentras lo que buscas **rápido y sin sorpresas** | Hay que **revisar y decidir** al momento de usar cada cosa |
| Si llega un producto nuevo, hay que **hacer un estante nuevo** (trabajo y permisos) | Si llega algo nuevo, **lo ponen en cualquier puesto** y listo |
| Ideal para **cuentas y balances** que no admiten errores | Ideal para **explorar** y guardar lo que aún no sabes cómo usarás |

Al terminar podrás explicar:
1. Qué hace **cada enfoque** cuando llega un dato **que no esperaba** y cuando llega un dato **malo**.
2. Por qué la flexibilidad **no es gratis**: el problema aparece **al usar** el dato.
3. Qué es un **«pantano de datos»** y cómo se evita con **etiquetas**.
4. Cómo decidir con **tres preguntas** simples (y por qué existe un «punto medio»: el *lakehouse*).

> 🔗 Si luego quieres ver el **experimento medido** (el mismo conjunto en DuckDB y en archivos JSON y Parquet), el modelo de **costo** y la guía de decisión completa: [cuaderno estándar](../02_Comparacion_Data_Warehouse_vs_Data_Lake.ipynb). Antes de esto: [el centro de acopio](01_Data_Lake_Dummies.ipynb).

---
## 1. Llega un Dato Inesperado y Llega un Dato Malo 📬

Simulamos las dos formas de guardar con cosas muy simples (listas y tablas pequeñas, datos **inventados**):

- **La tienda ordenada:** una función `guardar_en_tienda` que **revisa cada pedido** antes de dejarlo entrar: solo acepta las columnas que conoce (`producto`, `kilos`, `precio`) y exige que los kilos sean **mayores que cero**.
- **La plaza:** una lista `plaza` donde **todo se guarda sin revisar**.

Les mandamos tres pedidos: uno **normal**, uno con una **columna nueva** (`cupon`) y uno **malo** (kilos negativos).

Después queremos **sumar el dinero del día** (kilos × precio) y veremos **dónde aparece el problema** de cada enfoque: en la tienda la suma es directa; en la plaza hay que **revisar cada costal al momento de contar**, porque el pedido malo tiene **kilos negativos** y podría **restar** dinero sin que nadie se dé cuenta. Un lector **ingenuo** suma todo; uno **cuidadoso** aplica sus propias reglas **al leer** (el «esquema en lectura»).

In [ ]:
import numpy as np
import pandas as pd

COLUMNAS = ["producto", "kilos", "precio"]                       # lo unico que conoce la tienda ordenada


def guardar_en_tienda(tienda, pedido):
    """La tienda ordenada: revisa ANTES de guardar. Si algo no encaja, el pedido no entra."""
    if set(pedido) != set(COLUMNAS):
        raise ValueError(f"columnas distintas a las de la tienda: {sorted(set(pedido) ^ set(COLUMNAS))}")
    if pedido["kilos"] <= 0:
        raise ValueError("los kilos deben ser mayores que cero")
    tienda.append(pedido)


pedidos = [
    {"producto": "papa", "kilos": 10, "precio": 2200},                                 # normal
    {"producto": "papa", "kilos": 5, "precio": 2200, "cupon": "DESC10"},               # columna NUEVA
    {"producto": "queso", "kilos": -3, "precio": 18000},                               # dato MALO
]
tienda, plaza, rechazos = [], [], []
for p in pedidos:
    plaza.append(p)                                                                    # la plaza: todo entra
    try:
        guardar_en_tienda(tienda, p)
    except ValueError as e:
        rechazos.append(str(e))                                                        # la tienda: lo que no encaja se rechaza (con motivo)

print(f"Pedidos enviados: {len(pedidos)} | entraron a la tienda: {len(tienda)} | entraron a la plaza: {len(plaza)}")
print("Motivos de rechazo de la tienda:", rechazos)
assert len(tienda) == 1 and len(plaza) == 3 and len(rechazos) == 2
assert "cupon" in plaza[1] and "cupon" not in tienda[0]

# ---------------- La cuenta del dia: AQUI aparece el costo de la flexibilidad ----------------
def total(lista):
    return sum(p["kilos"] * p["precio"] for p in lista)


def total_cuidadoso(lista):
    """El lector cuidadoso pone sus reglas AL LEER: ignora lo que no tiene sentido."""
    return sum(p["kilos"] * p["precio"] for p in lista if p["kilos"] > 0)


en_tienda = total(tienda)
ingenuo = total(plaza)
cuidadoso = total_cuidadoso(plaza)
print(f"Tienda: {en_tienda} | plaza leida sin cuidado: {ingenuo} | plaza leida con cuidado: {cuidadoso}")
assert en_tienda == 22000 and ingenuo == 22000 + 5 * 2200 - 3 * 18000 and cuidadoso == 22000 + 5 * 2200
assert ingenuo != cuidadoso                                              # dos lectores, dos cifras distintas con los MISMOS datos

---
**Lección:** la **tienda** dejó entrar **un solo pedido** y rechazó dos **con su motivo**: protege la calidad, pero **tuvo que decir que no** a un dato que quizás era valioso (el cupón). La **plaza** guardó **los tres**, incluido el malo. ¿Gratis? No: con los **mismos datos de la plaza**, el lector ingenuo obtuvo **una cifra** y el cuidadoso **otra**: cuando el esquema se aplica **al leer**, **cada quien puede interpretar distinto**. En la tienda eso no pasa, porque el dato malo **nunca entró**. Por eso los lagos serios usan **zonas** (cruda y curada), **reglas escritas** y **pruebas**, y por eso existe el **punto medio** (*lakehouse*): archivos flexibles de la plaza **más** una capa que hace cumplir reglas como la tienda. (Ese punto medio es un **concepto**: aquí no lo instalamos.)

---

## 2. El Pantano: una Plaza sin Etiquetas 🐊

Una plaza **sin rótulos** se vuelve un **pantano**: hay de todo, pero nadie sabe **qué es, de quién es ni si ya hay otro igual**. Contamos 10 «puestos» (datos inventados): solo 3 tienen etiqueta. La **medida de salud** del lago es qué porcentaje está **etiquetado con dueño y descripción**. Luego aplicamos la regla: **no se acepta ningún puesto sin etiqueta completa**.

In [ ]:
puestos = pd.DataFrame({
    "puesto": [f"puesto_{i}" for i in range(1, 11)],
    "dueno": ["Rosa", None, None, "Beto", None, None, None, "Carla", None, None],
    "descripcion": ["papas criollas", None, "no se", "quesos", None, None, None, "frutas", None, ""],
})


def etiquetado(df):
    return df[df.dueno.notna() & df.descripcion.notna() & (df.descripcion.str.len() > 5)]


antes = len(etiquetado(puestos)) / len(puestos)
print(f"Puestos con etiqueta completa: {len(etiquetado(puestos))} de {len(puestos)} ({antes:.0%})")


def aceptar(df, nuevo):
    """La regla: no se acepta un puesto sin dueño ni descripcion de mas de 5 letras."""
    if not nuevo.get("dueno") or len(nuevo.get("descripcion") or "") <= 5:
        raise ValueError("puesto sin etiqueta completa")
    return pd.concat([df, pd.DataFrame([nuevo])], ignore_index=True)


try:
    puestos = aceptar(puestos, {"puesto": "puesto_11", "dueno": None, "descripcion": "sin dueño"})
    rechazado = False
except ValueError:
    rechazado = True
puestos = aceptar(puestos, {"puesto": "puesto_11", "dueno": "Ana", "descripcion": "arepas de maíz"})
despues = len(etiquetado(puestos)) / len(puestos)
print(f"Con la regla, el nuevo puesto entra solo si esta etiquetado: ahora {len(etiquetado(puestos))} de {len(puestos)} ({despues:.0%})")
assert len(etiquetado(puestos.iloc[:10])) == 3 and rechazado and len(puestos) == 11

---
**Lección:** una regla muy simple (**«sin etiqueta no entra»**) evita que el pantano **siga creciendo**; los 7 puestos viejos **siguen sin etiqueta** y alguien (una **persona**) tiene que **ponerles dueño y descripción**: el código solo **obliga** y **mide**. (Y ojo con la trampa: una etiqueta como «no se» **pasa** una regla ingenua; por eso la regla exige más de 5 letras, pero **ninguna regla automática** reemplaza el criterio humano.)

---

## 3. Tres Preguntas para Decidir 🧭

Una guía **muy simplificada** (no es una ley, es para **conversar**):

1. ¿Tus datos son **casi todos tablas** y los usarán sobre todo **analistas de negocio**, con cifras que **no admiten errores**? → **Almacén**.
2. ¿Tienes **fotos, textos, sensores** o datos que **aún no sabes cómo usar**, y un equipo técnico que pueda **ponerles orden**? → **Lago**.
3. ¿Necesitas **las dos cosas** (tablas confiables **y** datos variados)? → **Punto medio** (*lakehouse*).

In [ ]:
def guia(casi_todo_tablas, datos_variados, usuarios_de_negocio, equipo_tecnico_para_ordenar):
    """Tres preguntas -> una sugerencia (muy simplificada)."""
    if casi_todo_tablas and usuarios_de_negocio and not datos_variados:
        return "almacén"
    if datos_variados and not usuarios_de_negocio:
        return "lago" if equipo_tecnico_para_ordenar else "lago con mucho cuidado (riesgo de pantano)"
    return "punto medio (lakehouse)"


casos = {
    "Contabilidad de una tienda": guia(True, False, True, False),
    "Fotos de cultivos y sensores de un grupo de ingenieros": guia(False, True, False, True),
    "Una cooperativa con ventas, fotos y analistas": guia(False, True, True, True),
    "Fotos y sensores sin equipo técnico": guia(False, True, False, False),
}
for nombre, sug in casos.items():
    print(f"{nombre:56s} -> {sug}")
assert list(casos.values()) == ["almacén", "lago", "punto medio (lakehouse)", "lago con mucho cuidado (riesgo de pantano)"]

---
**Lección:** no hay una respuesta única; la **mejor** depende de **quién usa el dato, qué tipo de dato es y cuánto orden puedes mantener**. Y lo más importante: el almacén y el lago **no se excluyen**; muchas organizaciones usan **los dos**.

---

## Resumen en una frase 🎯

> **El almacén es la tienda ordenada (nada entra si no encaja: confiable pero rígida) y el lago es la plaza de mercado (todo entra: flexible, pero el problema aparece al usar el dato y, sin etiquetas, se vuelve pantano); el punto medio busca lo mejor de los dos.**

### Lo que aprendiste hoy:
- ✅ **Dato inesperado y dato malo:** la tienda **rechaza con motivo** (1 de 3 pedidos entró); la plaza **guarda todo** (3 de 3).
- ✅ **El costo de la flexibilidad:** con los mismos datos, dos lectores obtuvieron **dos cifras distintas** (el esquema se aplica **al leer**).
- ✅ **Pantano:** sin etiquetas (dueño y descripción) nadie sabe qué hay; la regla **«sin etiqueta no entra»** frena el crecimiento, pero **ordenar lo viejo requiere personas**.
- ✅ **Decidir:** según el **tipo de dato**, **quién lo usa** y **cuánto orden** puedes mantener; el punto medio (*lakehouse*) combina ambos mundos.

> 🎓 **Siguiente paso:** el [cuaderno estándar](../02_Comparacion_Data_Warehouse_vs_Data_Lake.ipynb) **mide** el mismo conjunto de datos en un almacén y en un lago (rapidez, rigidez y costo de cambio) y calcula un **punto de equilibrio de costos** con supuestos explícitos. Después, el taller [`../../homeworks/02_Data_Warehouse_Data_Lake_Hands_On.ipynb`](../../homeworks/02_Data_Warehouse_Data_Lake_Hands_On.ipynb) *(se publica aparte)*.

---
##### 🛠️ Práctica 1: Una regla más para la tienda: precios positivos

Con la función `guardar_en_tienda(tienda, pedido)` y la lista `tienda`:

1. Escribe una función `guardar_en_tienda_2(tienda, pedido)` que haga **lo mismo** que `guardar_en_tienda` **y además** rechace los pedidos con `precio <= 0` (con `ValueError`).
2. Pruébala con un pedido normal (`{"producto": "leche", "kilos": 2, "precio": 2800}`) y con uno de precio `0`.
3. Comprueba con `assert` que el normal **entra** a una tienda nueva `tienda_2` (que empieza vacía) y que el de precio `0` **no entra** (que `ValueError` se levantó).

In [ ]:
# Escribe tu código aquí

# def guardar_en_tienda_2(tienda, pedido):
#     ...
# tienda_2 = []


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
def guardar_en_tienda_2(tienda, pedido):
    guardar_en_tienda([], pedido)                              # reutiliza las revisiones de antes (columnas y kilos)
    if pedido["precio"] <= 0:
        raise ValueError("el precio debe ser mayor que cero")
    tienda.append(pedido)


tienda_2, levanto_error = [], False
guardar_en_tienda_2(tienda_2, {"producto": "leche", "kilos": 2, "precio": 2800})
try:
    guardar_en_tienda_2(tienda_2, {"producto": "leche", "kilos": 2, "precio": 0})
except ValueError:
    levanto_error = True

assert len(tienda_2) == 1 and levanto_error
print("Entró 1 pedido; el de precio 0 fue rechazado.")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. En tus palabras: ¿qué ganó la **tienda** al rechazar el pedido con kilos negativos y qué **perdió** al rechazar el del cupón? ¿Cómo podría la tienda **no perder** el cupón sin dejar de ser ordenada?
2. Con los mismos datos de la plaza, el lector ingenuo y el cuidadoso obtuvieron **cifras distintas**. ¿Cómo evitarías que dos analistas de una empresa lleguen a números diferentes?
3. Piensa en una organización que conozcas (una alcaldía, una cooperativa, un colegio). Con las **tres preguntas**, ¿qué le recomendarías: tienda, plaza o punto medio? ¿Por qué?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>